<a href="https://colab.research.google.com/github/dhwlxor/My-Ropo/blob/main/%EB%84%A4%ED%8A%B8%EC%9B%8C%ED%81%AC_%EC%99%B8%EB%B6%80%EB%A7%9D_%EC%86%8D%EB%8F%84_%EC%B2%B4%ED%81%AC_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install numpy
!pip install pandas
!pip install matplotlib
!pip install speedtest-cli

import speedtest
import time
import subprocess
import re
import ssl  # SSL 설정을 위해 추가

def get_physical_link_speed():
    try:
        cmd = "powershell \"Get-NetAdapter | Where-Object { $_.Status -eq 'Up' } | Select-Object -Property LinkSpeed\""
        result = subprocess.check_output(cmd, shell=True).decode('cp949', errors='ignore')
        speed_val = re.search(r'(\d+)', result)
        if speed_val:
            val = int(speed_val.group(1))
            return val * 1000 if "Gbps" in result else val
    except: return 0
    return 0

def main():
    # [핵심] SSL 인증서 검증을 무시하도록 전역 설정 (시간 오류 해결)
    ssl._create_default_https_context = ssl._create_unverified_context

    print("-" * 60)
    print("    🌐 네트워크 진단 테스트 (v6_SSL_Fix)")
    print("-" * 60)

    # 1. 물리 링크 속도 체크
    link_speed = get_physical_link_speed()
    print(f"\n[1] 물리적 연결 상태: {link_speed} Mbps (Link Speed)")

    # 2. 실제 외부망 속도 측정
    print("\n[2] 외부망 실제 속도 측정 중... (약 30초 소요)")
    try:
        # secure=True를 유지하되 위에서 SSL 검증을 껐으므로 403과 SSL 에러를 동시에 방지합니다.
        st = speedtest.Speedtest(secure=True)

        print("    - 서버 탐색 중...")
        st.get_best_server()

        print("    - 다운로드 측정 중...")
        d_speed = st.download() / 1_000_000

        print("    - 업로드 측정 중...")
        u_speed = st.upload() / 1_000_000

        print(f"\n    - 실제 다운로드: {d_speed:.2f} Mbps")
        print(f"    - 실제 업로드  : {u_speed:.2f} Mbps")

        # 3. 인프라 효율성 진단
        print("\n[3] 인프라 효율성 진단")
        efficiency = (d_speed / link_speed) * 100 if link_speed > 0 else 0
        print(f"    - 망 효율성: {efficiency:.1f}%")

        if link_speed >= 1000:
            if d_speed < 100:
                print("    ❌ [심각] 기가비트 망임에도 속도가 매우 낮습니다.")
            else:
                print("    ✅ [정상] 기가비트 망 성능이 원활합니다.")

        # 4. 최종 판정
        print("\n[4] 진단결과")
        if u_speed > 70:
            print("    🔵 생산 라인 운용 가능 (양호)")
        else:
            print("    🔴 속도가 낮습니다. 관리자 문의 요망")

    except Exception as e:
        print(f"\n❌ 측정 오류 발생: {e}")
        print("💡 팁: PC의 '날짜 및 시간'이 현재 시간과 맞는지 확인해 보세요.")

if __name__ == "__main__":
    main()
    input("\n종료하려면 엔터키를 누르세요...")


------------------------------------------------------------
    🌐 네트워크 진단 테스트 (v6_SSL_Fix)
------------------------------------------------------------

[1] 물리적 연결 상태: 0 Mbps (Link Speed)

[2] 외부망 실제 속도 측정 중... (약 30초 소요)
    - 서버 탐색 중...
    - 다운로드 측정 중...
    - 업로드 측정 중...

    - 실제 다운로드: 1975.28 Mbps
    - 실제 업로드  : 749.11 Mbps

[3] 인프라 효율성 진단
    - 망 효율성: 0.0%

[4] 진단결과
    🔵 생산 라인 운용 가능 (양호)
